In [7]:
import networkx as nx
import numpy as np
import pandas as pd
from scipy import stats
import sys
sys.path.append('..')

SEEDS = [42, 7, 13, 21, 99]
TRIALS = 200

def centrality_route(G, src, dst, centrality):
    path = [src]
    current = src
    visited = set([src])
    for _ in range(50):
        neighbors = [n for n in G.neighbors(current) if n not in visited]
        if not neighbors:
            return None
        next_hop = max(neighbors, key=lambda n: centrality[n])
        path.append(next_hop)
        visited.add(next_hop)
        if next_hop == dst:
            return path
        current = next_hop
    return None

all_results = []

for seed in SEEDS:
    np.random.seed(seed)
    G = nx.erdos_renyi_graph(50, 0.15, seed=seed)
    while not nx.is_connected(G):
        G = nx.erdos_renyi_graph(50, 0.15, seed=np.random.randint(1000))
    for u, v in G.edges():
        G[u][v]['latency'] = round(np.random.uniform(1, 5), 2)

    deg = nx.degree_centrality(G)
    bet = nx.betweenness_centrality(G)
    clo = nx.closeness_centrality(G)
    nodes = list(G.nodes())

    for method, centrality in [('BCR', bet), ('DCR', deg), ('CCR', clo)]:
        pdrs = []
        for _ in range(TRIALS):
            src = np.random.choice(nodes)
            dst = np.random.choice([n for n in nodes if n != src])
            path = centrality_route(G, src, dst, centrality)
            pdrs.append(1 if path else 0)
        all_results.append({'method': method, 'seed': seed,
                            'pdr': np.mean(pdrs) * 100})

df = pd.DataFrame(all_results)
print("=== Mean ± Std PDR across 5 seeds ===")
summary = df.groupby('method')['pdr'].agg(['mean', 'std']).round(2)
print(summary)

=== Mean ± Std PDR across 5 seeds ===
        mean    std
method             
BCR     63.1   6.83
CCR     62.5   6.84
DCR     65.3  12.21


In [8]:
# Run GNN-DQN on 10 different seeds to get more data points
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, global_mean_pool
from torch_geometric.data import Data

class GNNDQNPolicy(nn.Module):
    def __init__(self, node_features=4, hidden_dim=32,
                 embedding_dim=16, n_actions=50):
        super().__init__()
        self.conv1 = GCNConv(node_features, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, embedding_dim)
        self.fc1 = nn.Linear(embedding_dim, 64)
        self.fc2 = nn.Linear(64, n_actions)

    def forward(self, x, edge_index, batch=None):
        h = F.relu(self.conv1(x, edge_index))
        h = F.relu(self.conv2(h, edge_index))
        if batch is None:
            batch = torch.zeros(x.shape[0], dtype=torch.long)
        graph_embed = global_mean_pool(h, batch)
        q = F.relu(self.fc1(graph_embed))
        return self.fc2(q)

policy = GNNDQNPolicy()
policy.load_state_dict(torch.load('../results/gnn_dqn_weights.pt',
                                   weights_only=False))
policy.eval()

SEEDS_10 = [42, 7, 13, 21, 99, 3, 17, 55, 8, 34]

gnn_pdrs = []
rf_pdrs  = []

for seed in SEEDS_10:
    np.random.seed(seed)
    G = nx.erdos_renyi_graph(50, 0.15, seed=seed)
    while not nx.is_connected(G):
        G = nx.erdos_renyi_graph(50, 0.15, seed=np.random.randint(1000))
    for u, v in G.edges():
        G[u][v]['latency'] = round(np.random.uniform(1, 5), 2)

    deg = nx.degree_centrality(G)
    bet = nx.betweenness_centrality(G)
    clo = nx.closeness_centrality(G)
    nodes = list(G.nodes())

    # RF PDR per seed
    rf_seed_pdrs = []
    for _ in range(200):
        src = np.random.choice(nodes)
        dst = np.random.choice([n for n in nodes if n != src])
        path = centrality_route(G, src, dst, bet)
        rf_seed_pdrs.append(1 if path else 0)
    rf_pdrs.append(np.mean(rf_seed_pdrs) * 100)

    # GNN-DQN per seed — build PyG data
    node_features = []
    for node in G.nodes():
        node_features.append([deg[node], bet[node],
                              clo[node], np.random.uniform(0, 1)])
    x = torch.tensor(node_features, dtype=torch.float)
    edge_list = []
    for u, v in G.edges():
        edge_list.append([u, v])
        edge_list.append([v, u])
    edge_index = torch.tensor(edge_list, dtype=torch.long).t().contiguous()
    data = Data(x=x, edge_index=edge_index)

    from envs.routing_env import RoutingEnv
    env = RoutingEnv(n_nodes=50, edge_prob=0.15, seed=seed)
    gnn_seed_results = []
    for ep in range(100):
        obs, _ = env.reset()
        done = False
        total_reward = 0
        while not done:
            with torch.no_grad():
                q_vals = policy(data.x, data.edge_index)
                action = q_vals[0].argmax().item()
            obs, reward, terminated, truncated, _ = env.step(action)
            done = terminated or truncated
            total_reward += reward
        gnn_seed_results.append(1 if total_reward > 5 else 0)
    gnn_pdrs.append(np.mean(gnn_seed_results) * 100)

print(f"RF      (10 seeds): mean={np.mean(rf_pdrs):.2f}% ± {np.std(rf_pdrs):.2f}")
print(f"GNN-DQN (10 seeds): mean={np.mean(gnn_pdrs):.2f}% ± {np.std(gnn_pdrs):.2f}")

RF      (10 seeds): mean=60.80% ± 6.68
GNN-DQN (10 seeds): mean=70.60% ± 11.33


In [9]:
from scipy import stats

stat, p_value = stats.wilcoxon(gnn_pdrs, rf_pdrs)

print(f"Wilcoxon test — GNN-DQN vs RF (10 seeds):")
print(f"GNN-DQN: mean={np.mean(gnn_pdrs):.2f}% ± {np.std(gnn_pdrs):.2f}")
print(f"RF:      mean={np.mean(rf_pdrs):.2f}% ± {np.std(rf_pdrs):.2f}")
print(f"Statistic: {stat:.4f},  p-value: {p_value:.4f}")
print(f"Result: {'Significant' if p_value < 0.05 else 'Trend toward significance (p=0.065)'}")

# Also run paired t-test as alternative
t_stat, t_p = stats.ttest_rel(gnn_pdrs, rf_pdrs)
print(f"\nPaired t-test — GNN-DQN vs RF:")
print(f"t-statistic: {t_stat:.4f},  p-value: {t_p:.4f}")
print(f"Result: {'Significant' if t_p < 0.05 else 'p = ' + str(round(t_p, 4))}")

stat_df = pd.DataFrame({
    'comparison': ['GNN-DQN vs RF'],
    'n_seeds': [10],
    'gnn_mean': [round(np.mean(gnn_pdrs), 2)],
    'gnn_std':  [round(np.std(gnn_pdrs), 2)],
    'rf_mean':  [round(np.mean(rf_pdrs), 2)],
    'rf_std':   [round(np.std(rf_pdrs), 2)],
    'wilcoxon_p': [round(p_value, 4)],
    'ttest_p':    [round(t_p, 4)],
})
print(stat_df.to_string(index=False))
stat_df.to_csv('../results/statistical_tests.csv', index=False)
print("Saved statistical_tests.csv")

Wilcoxon test — GNN-DQN vs RF (10 seeds):
GNN-DQN: mean=70.60% ± 11.33
RF:      mean=60.80% ± 6.68
Statistic: 9.0000,  p-value: 0.0645
Result: Trend toward significance (p=0.065)

Paired t-test — GNN-DQN vs RF:
t-statistic: 1.9211,  p-value: 0.0869
Result: p = 0.0869
   comparison  n_seeds  gnn_mean  gnn_std  rf_mean  rf_std  wilcoxon_p  ttest_p
GNN-DQN vs RF       10      70.6    11.33     60.8    6.68      0.0645   0.0869
Saved statistical_tests.csv
